## Hypothesis Testing

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind

df = pd.read_csv(r"..\data\preprocessed_earthquake_data.csv")

#### 1. Magnitude Difference by Earthquake Type
Hypothesis: The mean earthquake magnitude differs significantly between reviewed and non-reviewed events.

In [7]:
# Select relevant groups, drop missing values
group1 = df[df['status'] == 'reviewed']['mag'].dropna()
group2 = df[df['status'] != 'reviewed']['mag'].dropna()

ttest_result = ttest_ind(group1, group2, equal_var=False)
print('t-test statistic:', ttest_result.statistic)
print('p-value:', ttest_result.pvalue)

t-test statistic: -1.0748565111189015
p-value: 0.28494966597262317


**Statement**:The t-test result for Hypothesis-1 yields a t-statistic of -1.07 and a p-value of 0.285. Since the p-value is greater than the common significance threshold of 0.05, there is not enough evidence to reject the null hypothesis. Thus, the mean earthquake magnitude does not differ significantly between reviewed and non-reviewed events in your data.

#### 2. Depth Difference by Magnitude Type
Hypothesis: The mean depth of earthquakes significantly varies between 'mb' and 'mw' magnitude type categories.

In [8]:
group1 = df[df['magType'] == 'mb']['depth'].dropna()
group2 = df[df['magType'] == 'mw']['depth'].dropna()

ttest_result = ttest_ind(group1, group2, equal_var=False)
print('t-test statistic:', ttest_result.statistic)
print('p-value:', ttest_result.pvalue)

t-test statistic: 22.514758657461467
p-value: 8.933021714439036e-112


**Statement**: The t-test result for this hypothesis yields a t-test statistic of 22.51 and a p-value of approximately $8.93 \times 10^{-112}$. Since this p-value is extremely small and far below the conventional significance threshold of 0.05, there is extremely strong evidence to reject the null hypothesis. Therefore, the mean depth of earthquakes is significantly different between `mb` and `mw` magnitude type events in your dataset.

#### 3. Magnitude Consistency Across Months
Hypothesis: There is no significant association between the month of occurrence and the categorical bins of earthquake magnitude (such as low, medium, high).

In [9]:
# Bin magnitudes into categories
df['mag_bin'] = pd.cut(df['mag'], bins=[0, 3.5, 5.0, 6.5, 9], labels=['Low', 'Medium', 'High', 'Very High'])

# Contingency table: mag_bin vs Month
contingency = pd.crosstab(df['mag_bin'], df['Month'])
chi2, p, dof, expected = chi2_contingency(contingency)
print('Chi-square statistic:', chi2)
print('Degrees of freedom:', dof)
print('p-value:', p)

Chi-square statistic: 46.542726462177434
Degrees of freedom: 33
p-value: 0.05922370884750974


**Statement**: The chi-square test result for this hypothesis yields a chi-square statistic of 46.54, degrees of freedom of 33, and a p-value of approximately 0.0592. Since this p-value is slightly above the conventional significance threshold of 0.05, there is insufficient evidence to reject the null hypothesis at the 5% level. Therefore, the data does not provide strong enough evidence to conclude that earthquake magnitude category and month of occurrence are dependent variables.

#### 4. DayOfWeek Consistency Across Event Type
Hypothesis: The type of event ('type' column: earthquake, quarry blast, etc.) is independent of the day of week
(DayOfWeek) on which it occurred.

In [10]:
contingency = pd.crosstab(df['type'], df['DayOfWeek'])
chi2, p, dof, expected = chi2_contingency(contingency)
print('Chi-square statistic:', chi2)
print('Degrees of freedom:', dof)
print('p-value:', p)

Chi-square statistic: 92.16992465854213
Degrees of freedom: 36
p-value: 8.07286408381138e-07


**Statement**: The chi-square test result for this hypothesis yields a chi-square statistic of 92.17, degrees of freedom of 36, and a p-value of approximately $8.072 \times 10^{-07}$
 . Since this p-value is far below the conventional significance threshold of 0.05, there is very strong evidence to reject the null hypothesis. Therefore, the day of the week and event type are significantly dependent, indicating a meaningful association between these categorical variables in your data.

#### 5. Magnitude Mean Difference by Year Decade
Hypothesis: The mean earthquake magnitude (mag) differs significantly between records from two different decades (e.g., 2000s vs 2010s).

In [11]:
df['decade'] = (df['Year'] // 10) * 10
dec1 = df[df['decade'] == 2000]['mag'].dropna()
dec2 = df[df['decade'] == 2010]['mag'].dropna()

ttest_result = ttest_ind(dec1, dec2, equal_var=False)
print('t-test statistic:', ttest_result.statistic)
print('p-value:', ttest_result.pvalue)

t-test statistic: 7.034358911416218
p-value: 2.0392846020245177e-12


**Statement**: The t-test result for this hypothesis yields a t-test statistic of 7.03 and a p-value of approximately $2.039 \times 10^{-12}$. Since this p-value is extremely small and far below the conventional significance threshold of 0.05, there is very strong evidence to reject the null hypothesis. Therefore, the mean earthquake magnitude differs significantly between the two compared decades in your dataset.

### Task:

- Suggest five additional hypothesis tests, based on the preprocessed earthquake dataset, that cover both numerical and categorical variables using t-tests and chi-square tests, to extend the depth of your analysis.

#### 6. Magnitude Difference by Review Status (Welch t-test)
**Hypothesis:** The average earthquake magnitude differs between **reviewed** and **non-reviewed** events.


In [12]:
from scipy import stats
import numpy as np

# Prepare groups
g1 = df.loc[df['status'].astype(str).str.lower() == 'reviewed', 'mag'].astype(float).dropna()
g2 = df.loc[df['status'].astype(str).str.lower() != 'reviewed', 'mag'].astype(float).dropna()

t_stat, p_val = (np.nan, np.nan)
if len(g1) >= 2 and len(g2) >= 2:
    t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=False, nan_policy='omit')

print("Test: Welch t-test (Magnitude ~ Review Status)")
print(f"n(reviewed)={len(g1)}, n(non-reviewed)={len(g2)}")
print(f"t-statistic={t_stat:.6g}, p-value={p_val:.6g}")

Test: Welch t-test (Magnitude ~ Review Status)
n(reviewed)=109744, n(non-reviewed)=104
t-statistic=-1.07486, p-value=0.28495


#### 7. Depth Difference Across Event Types (Top 2) (Welch t-test)
**Hypothesis:** The mean **depth** differs between the two most frequent **event types**.


In [13]:
type_counts = df['type'].astype(str).value_counts(dropna=True)
if len(type_counts) < 2:
    print("Not enough distinct event types for this test.")
else:
    t1, t2 = type_counts.index[:2].tolist()
    a = df.loc[df['type'].astype(str) == t1, 'depth'].astype(float).dropna().values
    b = df.loc[df['type'].astype(str) == t2, 'depth'].astype(float).dropna().values

    if len(a) >= 2 and len(b) >= 2:
        t_stat, p_val = stats.ttest_ind(a, b, equal_var=False, nan_policy='omit')
        print("Test: Welch t-test (Depth ~ Type (Top 2))")
        print(f"Types compared: {t1} vs {t2}")
        print(f"n({t1})={len(a)}, n({t2})={len(b)}")
        print(f"t-statistic={t_stat:.6g}, p-value={p_val:.6g}")
    else:
        print(f"Insufficient sample sizes for {t1} and/or {t2}.")

Test: Welch t-test (Depth ~ Type (Top 2))
Types compared: earthquake vs nuclear explosion
n(earthquake)=109356, n(nuclear explosion)=424
t-statistic=165.131, p-value=0


#### 8. Magnitude Difference by Hemisphere (Welch t-test)
**Hypothesis:** The average **magnitude** differs between the **Northern** and **Southern** hemispheres.


In [14]:
hemi = np.where(df['latitude'].astype(float) >= 0, 'North', 'South')
gN = df.loc[hemi == 'North', 'mag'].astype(float).dropna()
gS = df.loc[hemi == 'South', 'mag'].astype(float).dropna()

t_stat, p_val = (np.nan, np.nan)
if len(gN) >= 2 and len(gS) >= 2:
    t_stat, p_val = stats.ttest_ind(gN, gS, equal_var=False, nan_policy='omit')

print("Test: Welch t-test (Magnitude ~ Hemisphere)")
print(f"n(North)={len(gN)}, n(South)={len(gS)}")
print(f"t-statistic={t_stat:.6g}, p-value={p_val:.6g}")

Test: Welch t-test (Magnitude ~ Hemisphere)
n(North)=53830, n(South)=56018
t-statistic=11.8471, p-value=2.33124e-32


#### 9. Association Between Event Type and Month (Chi-square)
**Hypothesis:** **Event type** and **month** are **not independent** (i.e., certain types occur more in certain months).


In [15]:
from scipy.stats import chi2_contingency

cont = pd.crosstab(df['type'].astype(str), df['Month'])
chi2, p, dof, exp = chi2_contingency(cont)

print("Test: Chi-square (Type × Month)")
print("Degrees of freedom:", dof)
print("Chi-square statistic:", round(chi2, 6))
print("p-value:", round(p, 6))
print("\nObserved counts (head):")
print(cont.head())

Test: Chi-square (Type × Month)
Degrees of freedom: 66
Chi-square statistic: 287.684863
p-value: 0.0

Observed counts (head):
Month                1     2     3     4     5     6     7     8     9     10  \
type                                                                            
earthquake         8885  8170  9708  9102  9136  8743  9870  9602  8847  8934   
explosion             1     1     2     1     1     0     1     1     0     0   
landslide             0     0     0     0     0     0     0     2     0     0   
mine collapse         0     1     0     0     0     0     0     0     0     0   
nuclear explosion     8    24    24    31    33    46    46    38    42    57   

Month                11    12  
type                           
earthquake         8706  9653  
explosion             1     1  
landslide             0     0  
mine collapse         0     0  
nuclear explosion    31    44  


#### 10. Association Between Event Type and Hemisphere (Chi-square)
**Hypothesis:** The distribution of **event types** differs between the **Northern** and **Southern** hemispheres.


In [16]:
hemi = np.where(df['latitude'].astype(float) >= 0, 'North', 'South')
cont = pd.crosstab(pd.Series(hemi, name='Hemisphere'), df['type'].astype(str))

chi2, p, dof, exp = chi2_contingency(cont)
print("Test: Chi-square (Hemisphere × Type)")
print("Degrees of freedom:", dof)
print("Chi-square statistic:", round(chi2, 6))
print("p-value:", round(p, 6))
print("\nObserved counts:")
print(cont)

Test: Chi-square (Hemisphere × Type)
Degrees of freedom: 6
Chi-square statistic: 291.819036
p-value: 0.0

Observed counts:
type        earthquake  explosion  landslide  mine collapse  \
Hemisphere                                                    
North            53401          8          2              1   
South            55955          2          0              0   

type        nuclear explosion  rock burst  volcanic eruption  
Hemisphere                                                    
North                     364           1                 53  
South                      60           0                  1  
